# CA3 - Classification of Wine

#### Gr 30: Sondre Aasheim & Zuzanna Magdalena Smolinska

### Imports

In [ ]:
import numpy as np                      # for numerical operations
import pandas as pd                     # for reading the csv files and working with dataframes
import matplotlib.pyplot as plt         # for basic plotting
import seaborn as sns                   # for more advanced plotting

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import warnings 
warnings.filterwarnings('ignore')


### Reading the data

We use the provided files: `train.csv`  - containing 835 labelled wines that we use to train our models, and `test.csv` - containing 209 unlabelled wines that we use for the predicttion for the Kaggle Competition

In [ ]:
# reading the data (both files must be in the same folder as this notebook)
wine_train = pd.read_csv("train.csv")
wine_test = pd.read_csv("test.csv")
 
# printing the training data to get a look
print(wine_train)

In [ ]:
# printing the test data to get a look at it as well
print(wine_test)

In [ ]:
# to double check that we have the right files, and to display their size (rows, columns)
print("Training data size: ", wine_train.shape)
print("Test data size: ", wine_test.shape)

# list of all column names in the training data
print(wine_train.columns.tolist())


### Data exploration and visualisation

In [ ]:
# to display a visualisation of the various feautures 
wine_train.hist(figsize=(12, 10), bins=20)
plt.tight_layout()
plt.show()

# Displaying the outliers by directly showing where different samples cluster together, and by extension the outliers
plt.boxplot(wine_train)
plt.xticks([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], 
           ["fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar", "chlorides", 
            "free_sulfur_dioxide", "total_sulfur_dioxide", "density", "pH", "sulphates", "alcohol"], 
            rotation=90)
plt.show()

### Data cleaning

In [ ]:
# counting all missing values in the training and test data
# this should give 0, so there are no missing numbers or information

print(wine_train.isna().sum().sum())
print(wine_test.isna().sum().sum())

### Data preprocessing

In [ ]:
# Displaying the outliers by directly showing where different samples cluster together, and by extension the outliers
plt.boxplot(wine_train)
plt.xticks([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], 
           ["fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar", "chlorides", 
            "free_sulfur_dioxide", "total_sulfur_dioxide", "density", "pH", "sulphates", "alcohol"], 
            rotation=90)
plt.show()

As can be seen in the boxes, a lot of the data, even most if not all of some feautures falls outside of the "average" meaning removing the outliers will be harder than just removing those that fall outside the average. As there are still clear outliers.


**Discusion**:
In regards to outliers, there can be made an argument that in this instance it would be unecasary to remove them. As we do not know the standards of which these tests were carried out in. Or if these outliers are even that far fetched in regards to bigger samples of wine. They would also simply be filed away into the lower quality catagori if kept. 

In [ ]:
# Gives each item inside the dataframe a score based on how far it is from the mean through z = (x-mu)/sigma
z_scores = (wine_train - wine_train.mean()) / wine_train.std(ddof=0)

# Checks if the z score is above or under the average by 3 through using absolute values
outliers = z_scores.abs() > 3

print("Outliers per feature:")
print(outliers.sum())

In [ ]:
# We'll take a deeper look at residual sugar, chlorides, free sulfur dioxide and sulphates
# since they are much higher than the rest
plt.boxplot(wine_train["residual sugar"])
plt.show()

plt.boxplot(wine_train["chlorides"])
plt.show()

plt.boxplot(wine_train["free sulfur dioxide"])
plt.show()

plt.boxplot(wine_train["sulphates"])
plt.show()

**Discusion**:
there can be made another argument on wether these four should have the limits on their Z_scores heightened to reduce their outlier count. Though this would in turn produce a new variable in the form of ununinform fitting

In [ ]:
# Identify rows containing at least one outlier
outlier_rows = outliers.any(axis=1)

# To make sure we aren't removing too much data, which 73 from 835 is a acceptilbe number 
print(f"Rows to remove: {outlier_rows.sum()}")

# Filtering the data
wine_train_clean = wine_train.loc[~outlier_rows].copy()

In [ ]:
# Displaying the graphs again to visually confirm that the outliers have been removed. No need for thourougnes
wine_train_clean.hist(figsize=(12, 10), bins=20)
plt.tight_layout()
plt.show()

plt.boxplot(wine_train_clean)
plt.xticks([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], 
           ["fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar", "chlorides", 
            "free_sulfur_dioxide", "total_sulfur_dioxide", "density", "pH", "sulphates", "alcohol"], 
            rotation=90)
plt.show()

### Modelling

In this section we compare five different classifiers from the lectures: (1) Logistic Regression, (2) SVM, (3) k-Nearest Neighbours, (4) decision tree and (5) random forest.

We split our training data to measure accuracy of our models. We cannot do that on the test set, as it does not contain labels that would enable verification.

In this situation, we split the labelled training data into a training part (70%) and a validation part (30%). To ensure accuracy of this method, we repeat the verification for 50 different random splits and take the mean -> for every model.


### Model 1 - Logistic Regression

In [ ]:
"""
We define settings for all the classifiers we will be using.
In this way, we want to ensure comparability between the results of our tests.
"""
N_SPLITS = 50           # number of cross-validation splits; 50 gives a stable average
TEST_SIZE = 0.3         # proportion of the dataset to include in the test split

In [ ]:
features = [c for c in wine_train.columns if c not in ("quality", "Id")]
print(features)

X = wine_train[features]        # input: the 11 chemical properties of each wine
y = wine_train["quality"]       # answer: 1 = good, 0 = bad
X_test = wine_test[features]    # the Kaggle test wines (no answers), used only at the very end

print("X:", X.shape, " y:", y.shape, " X_test:", X_test.shape)

In [ ]:
"""
We try our different hyperparameter values. 
Smaller C means strong regularisation and a risk of underfitting.
Larger C means weak regularisation and a risk of overfitting.
"""
params_C_lr = [0.001, 0.01, 0.1, 1, 10, 100]

"""
We will use two different penalties for the logistic regression classifier.
L1 or L2 adds a penalty when the weights get big, which is a form of regularisation.
"""
penalties = ["l1", "l2"]

accArr_train_lr = np.zeros((len(penalties), len(params_C_lr)))
accArr_val_lr = np.zeros((len(penalties), len(params_C_lr)))
accArr_val_std_lr = np.zeros((len(penalties), len(params_C_lr)))


for p_ind, penalty in enumerate(penalties):
    for c_ind, c in enumerate(params_C_lr):

        accTrain_list, accVal_list = [], []
        for rs in range(N_SPLITS):

            # X: the features of each wine (alcohol, pH, etc), which are the model's input
            # y: the answer for each wine (good or bad), which is what the model should predict
            
            X_train, X_val, y_train, y_val = train_test_split(
                X, y, test_size=0.3, random_state=rs, stratify=y)

            sc = StandardScaler()
            sc.fit(X_train)                         # learn mean and std from the training part only
            X_train_sc = sc.transform(X_train)      # apply them to the training part
            X_val_sc = sc.transform(X_val)          # apply the same mean and std to the validation part

            # initialise and train logistic regression with this penalty and C
            lr = LogisticRegression(C=c, penalty=penalty, solver="saga", max_iter=10000)
            lr.fit(X_train_sc, y_train)

            # compute train and validation accuracy and collect them in lists
            accTrain_list.append(lr.score(X_train_sc, y_train))
            accVal_list.append(lr.score(X_val_sc, y_val))

        # average accuracies across all splits
        accArr_train_lr[p_ind, c_ind] = np.mean(accTrain_list)
        accArr_val_lr[p_ind, c_ind] = np.mean(accVal_list)


In [ ]:
# the results as a table (rows = penalty, columns = C)
print("Mean validation accuracy:")
print(pd.DataFrame(accArr_val_lr, index=penalties, columns=params_C_lr).round(4))

# only used to check for over/underfitting
print("\nMean train accuracy:")
print(pd.DataFrame(accArr_train_lr, index=penalties, columns=params_C_lr).round(4))


best_params = np.unravel_index(accArr_val_lr.argmax(), accArr_val_lr.shape)

best_lr = {"penalty": penalties[best_params[0]],
           "C": params_C_lr[best_params[1]],
           "acc": round(float(accArr_val_lr[best_params]), 4)}
print("\nBest Logistic Regression:", best_lr)

### Model 2 - Support Vector Machine (SVM)

SVM looks for the boundary between good and bad wines that has the widest possible margin, meaning the largest distance to the nearest wines of each class. Those nearest wines are called support vectors - they decide where the boundary goes.

With a kernel, the SVM can draw curved boundaries instead of only straight lines (logistic regression can only draw straight ones). We first compare three kernels, then tune the best one.

In [ ]:
kernels = {
    "linear":             {"kernel": "linear"},
    "polynomial (deg 3)": {"kernel": "poly", "degree": 3},
    "RBF":                {"kernel": "rbf"},
}

for name, settings in kernels.items():
    accVal_list = []

    for rs in range(N_SPLITS):
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=rs, stratify=y)

        sc = StandardScaler()
        sc.fit(X_train)
        X_train_sc = sc.transform(X_train)
        X_val_sc = sc.transform(X_val)

        svm_model = SVC(C=1, **settings)
        svm_model.fit(X_train_sc, y_train)
        accVal_list.append(svm_model.score(X_val_sc, y_val))

    print(f"SVM kernel = {name:<20} mean validation accuracy = {np.mean(accVal_list):.4f}")


In [ ]:
"""
Now we tune the hyperparameters of the RBF kernel, which is the best one.
We try every combination of `C` and `gamma`, as in the lecture example.
"""
params_C = np.arange(-1, 2.1, 0.5)
params_gamma = np.array([0.01, 0.03, 0.05, 0.1, 0.2, 0.3, 0.5])

accArr_train = np.zeros((len(params_C), len(params_gamma)))
accArr_val = np.zeros((len(params_C), len(params_gamma)))

for c_ind, c in enumerate(params_C):

    for gamma_ind, gamma_val in enumerate(params_gamma):

        accTrain_list = []
        accVal_list = []

        for rs in range(N_SPLITS):

            X_train, X_val, y_train, y_val = train_test_split(
                X, y, test_size=TEST_SIZE, random_state=rs, stratify=y)

            sc = StandardScaler()
            sc.fit(X_train)

            X_train_sc = sc.transform(X_train)
            X_val_sc = sc.transform(X_val)

            # training SVC with specific values for C and gamma
            SVC_rbf = SVC(kernel="rbf", C=10.**c, gamma=gamma_val, random_state=1)
            SVC_rbf.fit(X_train_sc, y_train)

            # collecting train accuracies in list
            accTrain_list.append(SVC_rbf.score(X_train_sc, y_train))

            # collecting validation accuracies in list
            accVal_list.append(SVC_rbf.score(X_val_sc, y_val))


        accArr_train[c_ind, gamma_ind] = np.mean(accTrain_list)
        accArr_val[c_ind, gamma_ind] = np.mean(accVal_list)

In [ ]:
"""
We plot heat maps of train and validation accuracies.
We base our choice of hyperparameters on the validation accuracy, not the training accuracy.
Brighter colors indicate higher accuracy, darker colors indicate lower accuracy.
"""
rowNames = ["10**{0:3.1f}".format(c) for c in params_C]
colNames = ["{0:4.2f}".format(g) for g in params_gamma]

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(pd.DataFrame(accArr_train, index=rowNames, columns=colNames),
            annot=True, fmt=".2f", vmax=1, ax=ax[0])
ax[0].set_xlabel("gamma"); ax[0].set_ylabel("C"); ax[0].set_title("Train data accuracies")

sns.heatmap(pd.DataFrame(accArr_val, index=rowNames, columns=colNames),
            annot=True, fmt=".3f", ax=ax[1])
ax[1].set_xlabel("gamma"); ax[1].set_ylabel("C"); ax[1].set_title("Validation data accuracies")

plt.tight_layout()
plt.show()

In [ ]:
# choosing the best hyperparameters based on the validation accuracy
best_params = np.unravel_index(accArr_val.argmax(), accArr_val.shape)

best_svm = {"C": round(float(10.**params_C[best_params[0]]), 3),
            "gamma": float(params_gamma[best_params[1]]),
            "acc": round(float(accArr_val[best_params]), 4)}
print("Best SVM:", best_svm)

### Final evaluation

### Kaggle submission

In [ ]:
#this was just a example provided by the profesor, so up for editing to better suit our work!!

y_test = model.predict(X_test2_scaled)
y_test = pd.DataFrame(y_test2, columns=["Quality"])
y_test.index.name = "ID"
y_test[['Quality']].to_csv("submission.csv")